In [0]:
from pyspark.sql.functions import to_timestamp, current_timestamp, monotonically_increasing_id, lit, col

In [0]:
catalog = 'labuser11612924_1758234044'

In [0]:
df = spark.read.table(f'{catalog}.bronze.customers')

In [0]:
df.display()

In [0]:
df = df.drop('_rescued_data', 'filename')
df = df.withColumnRenamed('ingesttime', 'create_date')

In [0]:
df = df.dropDuplicates(['customer_id'])

In [0]:
display(df.limit(10))

### Dividing New and Old Records

In [0]:
if spark.catalog.tableExists(f'{catalog}.silver.customers'):
  df_old = spark.sql(f"""select DimKeyCustomer, customer_id, create_date, update_date from {catalog}.bronze.customers""")

else:
    df_old = spark.sql(f"""select 0 DimKeyCustomer, 0 customer_id, 0 create_date, 0 update_date from {catalog}.bronze.customers where 1 = 0""")

In [0]:
# df_old.display()

### Renaming Columns of old_df

In [0]:
df_old = df_old.withColumnRenamed('dimKeyCustomer', 'old_dimKeyCustomer') \
              .withColumnRenamed('customer_id', 'old_customer_id') \
              .withColumnRenamed('create_date', 'old_create_date') \
              .withColumnRenamed('update_date', 'old_update_date') 
# df_old.display()

### Applying Joins with the Old Records

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')
df_join.display()

### Separating Old VS New Records

In [0]:
df_new = df_join.filter(df_join['old_dimkeyCustomer'].isNull())
display(df_new)

In [0]:
df_old = df_join.filter(df_join['old_dimkeyCustomer'].isNotNull())
display(df_old.limit(10))

### Preparing Df_old

In [0]:
# Dropping all the columsn that are not required
df_old = df_old.drop('old_customer_id', 'old_update_date', 'old_create_date')

#  Renaming "old_dimKeyCustomer" to "dimKeyCustomer"
df_old = df_old.withColumnRenamed('old_dimKeyCustomer', 'dimKeyCustomer')

#  Renaming "old_create_date" to "create_date"
df_old = df_old.withColumnRenamed("old_create_date", "create_date")
# df_old = df_old.withColumn("create_date", to_timestamp("create_date"))

#  Recreating "update_date" column with Current timestamp
df_old = df_old.withColumn("update_date", current_timestamp()) 

In [0]:
df_old.display()

### Preparing df_new

In [0]:
# Dropping all the columsn that are not required
df_new = df_new.drop('old_dimKeyCustomer', 'old_customer_id', 'old_update_date', 'old_create_date', '_rescued_data','filename')

#  Recreating "iupdate_date" & "created_date" with Current Timestamp
df_new = df_new.withColumnRenamed("ingesttime", 'create_date')
df_new = df_new.withColumn("update_date", current_timestamp()) 

In [0]:
display(df_new.limit(10))

### Surrogate Key - From 1

In [0]:
df_new = df_new.withColumn("dimKeyCustomer", monotonically_increasing_id() + lit(1))

In [0]:
display(df_new.limit(10))

### Adding Max Surrogate Key

In [0]:
if spark.catalog.tableExists(f'{catalog}.silver.customers'):
    df_max_surr = spark.sql(f"""SELECT MAX(dimKeyCustomer) as max_surrogate_key FROM {catalog}.silver.customers""")
    # Converting df_max_surr to Surrogate Key Variable
    max_surrogate_key = df_max_surr.collect()[0]['max_surrogate_key']
else:
    max_surrogate_key = 0

In [0]:
max_surrogate_key

In [0]:
df_new = df_new.withColumn('dimKeyCustomer',lit(max_surrogate_key) + col('dimKeyCustomer'))

In [0]:
df_new.display()

### Union of df_old & df_new

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

In [0]:
df_final.schema

In [0]:
# df = spark.read.format('parquet').load('/Volumes/project/bronze/raw_customers/data')
# display(df.limit(10))

In [0]:
df = df.drop("_rescued_data")

In [0]:
from pyspark.sql import functions as F

In [0]:
df = df.withColumn("domain", F.split("email","@")[1])
display(df.limit(10))

In [0]:
df_agg  = df.groupBy("domain").agg(F.count("customer_id").alias("total_amount")) \
    .orderBy("total_amount", ascending=False)

display(df_agg.limit(10))

In [0]:
df_gmail = df.filter(F.col("domain") == "gmail.com")
display(df_gmail.limit(10))

In [0]:
df = df.withColumn('full_name', F.concat('first_name',F.lit(' '),'last_name')).drop('first_name','last_name')
display(df.limit(10))

In [0]:
# df.write.format("delta").mode("overwrite").save("/Volumes/project/volumes/silver_customers_volume/data")

In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('project.silver.customers')

In [0]:
%sql
-- describe volume project.volumes.silver_customers_volume


In [0]:
# %sql
# CREATE OR REPLACE TABLE project.silver.customers
# AS 
# SELECT * FROM DELTA.`/Volumes/project/volumes/silver_customers_volume/data`

In [0]:
%sql
select * from project.silver.customers